In [ ]:
import requests
import pandas as pd
from datetime import datetime
import smtplib
from email.message import EmailMessage
import os

# === CONFIGURACIÓN ===
HUBSPOT_TOKEN = "pat-na1-cd6a882d-248a-45d8-99fb-6d1e71f77845"
HEADERS = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}",
    "Content-Type": "application/json"
}

EMAIL_REMITENTE = "info@zentralcom.com"
EMAIL_CONTRASENA = "ptF3dCC2TBQPmV"
EMAIL_DESTINATARIO = "iban.ibanez@zentralcom.com"

# === FUNCIONES AUXILIARES ===
def cargar_propietarios():
    owners = {}
    offset = 0
    has_more = True

    while has_more:
        url = f"https://api.hubapi.com/crm/v3/owners?limit=100&after={offset}"
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            break

        data = response.json()
        for owner in data.get("results", []):
            owner_id = str(owner.get("ownerId") or owner.get("id"))
            full_name = f"{owner.get('firstName', '')} {owner.get('lastName', '')}".strip()
            if not full_name:
                full_name = owner.get("email", "Desconocido")
            owners[owner_id] = full_name

        offset = data.get("paging", {}).get("next", {}).get("after")
        has_more = bool(offset)

    return owners

def cargar_etapas_y_pipelines():
    url = "https://api.hubapi.com/crm/v3/pipelines/deals"
    response = requests.get(url, headers=HEADERS)
    stage_map = {}
    pipeline_map = {}
    if response.status_code == 200:
        for pipeline in response.json().get("results", []):
            pipeline_id = pipeline["id"]
            pipeline_map[pipeline_id] = pipeline["label"]
            for stage in pipeline["stages"]:
                stage_map[stage["id"]] = stage["label"]
    return stage_map, pipeline_map

def obtener_ultima_actividad(deal_id):
    url = f"https://api.hubapi.com/engagements/v1/engagements/associated/deal/{deal_id}/paged?limit=100"
    response = requests.get(url, headers=HEADERS)
    ultima_fecha = None
    tipo_actividad = ""
    tipos_validos = {"CALL", "EMAIL", "MEETING", "TASK", "NOTE"}
    ahora = datetime.now()
    if response.status_code == 200:
        data = response.json()
        for item in data.get("results", []):
            tipo = item.get("engagement", {}).get("type")
            ts = item.get("engagement", {}).get("timestamp")
            if tipo in tipos_validos and ts:
                fecha = datetime.fromtimestamp(ts / 1000)
                if fecha <= ahora and (not ultima_fecha or fecha > ultima_fecha):
                    ultima_fecha = fecha
                    tipo_actividad = tipo
    return ultima_fecha, tipo_actividad

def generar_informe_multiple_negocios(limit=None):
    print(f"📥 Obteniendo hasta {limit} negocios desde HubSpot...")

    stage_map, pipeline_map = cargar_etapas_y_pipelines()
    owner_map = {
        "74753477": "Iñigo Mangas Insausti",
        "75326442": "Jose Ramon Mendibil",
        "75326441": "Alejandro Alzas Moreno",
        "102123176": "Iban Ibañez",
        "75887314": "Onura Onura",
        "75631012": "Rafael Quintanilla"
    }

    all_deals = []
    url = "https://api.hubapi.com/crm/v3/objects/deals"
    params = {
        "limit": 100,
        "properties": ",".join([
            "dealname", "dealstage", "pipeline", "createdate",
            "hubspot_owner_id", "hs_lastmodifieddate",
            "hs_object_id", "familias_de_interes", "closed_won_reason"
        ]),
        "associations": "companies,contacts"
    }

    all_data = []
    while url:
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code != 200:
            print("❌ Error al obtener negocios:", response.status_code)
            return None

        data = response.json()
        all_data.extend(data.get("results", []))

        paging = data.get("paging", {}).get("next", {}).get("after")
        if paging:
            params["after"] = paging
        else:
            break

    for deal in all_data:
        props = deal.get("properties", {})
        deal_id = props.get("hs_object_id")
        owner_id_raw = str(props.get("hubspot_owner_id"))
        propietario = owner_map.get(owner_id_raw, owner_id_raw)

        fecha_creacion_cruda = props.get("createdate")
        fecha_creacion = ""
        if fecha_creacion_cruda:
            fecha_creacion_dt = datetime.fromisoformat(fecha_creacion_cruda.replace("Z", ""))
            fecha_creacion = fecha_creacion_dt.strftime("%Y-%m-%d %H:%M")
        else:
            fecha_creacion_dt = None

        fecha_actividad_dt, tipo_actividad = obtener_ultima_actividad(deal_id)
        if not fecha_actividad_dt and fecha_creacion_dt:
            fecha_actividad_dt = fecha_creacion_dt

        fecha_actividad = fecha_actividad_dt.strftime("%Y-%m-%d %H:%M") if fecha_actividad_dt else ""
        dias_sin_actividad = (datetime.today() - fecha_actividad_dt).days if fecha_actividad_dt else ""

        all_deals.append({
            "Nombre del negocio": props.get("dealname"),
            "Etapa del negocio": stage_map.get(props.get("dealstage"), props.get("dealstage")),
            "Pipeline": pipeline_map.get(props.get("pipeline"), props.get("pipeline")),
            "Propietario del negocio": propietario,
            "Fecha de creación": fecha_creacion,
            "Última actividad": fecha_actividad,
            "Tipo de última actividad": tipo_actividad,
            "Días sin actividad": dias_sin_actividad,
            "Familias de interes": props.get("familias_de_interes", "")
        })

    df = pd.DataFrame(all_deals)
    fecha = datetime.now().strftime("%Y-%m-%d")
    archivo = f"informe_hubspot_{fecha}.xlsx"
    df.to_excel(archivo, index=False)
    print(f"✅ Informe generado: {archivo}")
    return archivo

def enviar_email(archivo):
    msg = EmailMessage()
    msg["Subject"] = "Informe HubSpot"
    msg["From"] = EMAIL_REMITENTE
    msg["To"] = EMAIL_DESTINATARIO
    msg.set_content("Adjunto el informe de negocios extraído desde HubSpot.")

    with open(archivo, "rb") as f:
        contenido = f.read()
        msg.add_attachment(contenido, maintype="application", subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet", filename=os.path.basename(archivo))

    with smtplib.SMTP("smtp.office365.com", 587) as smtp:
        smtp.starttls()
        smtp.login(EMAIL_REMITENTE, EMAIL_CONTRASENA)
        smtp.send_message(msg)

    print("📧 Correo enviado con el informe adjunto.")

# === EJECUTAR DIRECTAMENTE ===
if __name__ == "__main__":
    archivo = generar_informe_multiple_negocios()
    if archivo:
        enviar_email(archivo)










📥 Obteniendo hasta None negocios desde HubSpot...
✅ Informe generado: informe_hubspot_2025-05-29.xlsx
📧 Correo enviado con el informe adjunto.
